# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring** — with a binary classifier working underneath it as one
component, not as the deliverable itself.

My own `notebooks/01` and `02` runs trained a classifier (logistic regression / decision tree /
random forest) on a binary label, `is_declining_label`. But per the lane guide, the thing this
lane actually ships is `final_refresh_score` — a blend of that model's probability and the
transparent baseline score — used only to **order** pages so a reviewer with limited time works
top-down. My own `w01` notebook already named the decision this improves as *"which pages to
review first,"* not *"is this page declining, yes/no."*

The `framing-ml-problems` skill frames task type off the decision, not the internal mechanism:
*"Which ones first?"* maps to ranking/scoring, target = a priority score, metric = precision@K.
Nobody in this workflow reads a bare "declining: yes" flag — the ranked queue is what a reviewer
actually opens, and landing at rank 3 vs rank 300 on that queue is the entire point.

So the classifier is a **component**, not the task: it produces a probability that becomes one
input to the score. The cell below shows why this isn't just semantics — the metric that was
actually validated on my own pipeline run, Precision@50, is a ranking metric, and it moves very
differently from the classifier's own diagnostics (ROC AUC / average precision).

In [1]:
# Real numbers from this repo's committed pipeline run: outputs/model_report.md
# ROC AUC / avg precision are classifier diagnostics (how well the model separates the proxy
# label anywhere in the data). Precision@50 is the metric that matches the actual review-queue
# decision: "of the top 50 pages the system ranks first, how many were actually worth it?"

model_comparison = {
    "baseline_rules":      {"roc_auc": 0.627, "avg_precision": 0.468, "precision_at_50": 0.240},
    "logistic_regression": {"roc_auc": 0.700, "avg_precision": 0.522, "precision_at_50": 0.400},
    "decision_tree":       {"roc_auc": 0.742, "avg_precision": 0.575, "precision_at_50": 0.540},
    "random_forest":       {"roc_auc": 0.750, "avg_precision": 0.618, "precision_at_50": 0.740},
}

print(f"{'model':<20}{'ROC AUC':>10}{'avg precision':>16}{'Precision@50':>16}")
for name, m in model_comparison.items():
    print(f"{name:<20}{m['roc_auc']:>10.3f}{m['avg_precision']:>16.3f}{m['precision_at_50']:>16.3f}")

print()
print("ROC AUC barely moves between the tree (0.742) and the random forest (0.750) -- as pure")
print("classifiers they look almost the same. But Precision@50, the number tied to the real")
print("decision, jumps from 0.540 to 0.740. That gap is the argument for calling this lane a")
print("ranking/scoring task: what matters lives in the ordering of the queue, not in the")
print("classifier's raw separability.")


model                  ROC AUC   avg precision    Precision@50
baseline_rules           0.627           0.468           0.240
logistic_regression      0.700           0.522           0.400
decision_tree            0.742           0.575           0.540
random_forest            0.750           0.618           0.740

ROC AUC barely moves between the tree (0.742) and the random forest (0.750) -- as pure
classifiers they look almost the same. But Precision@50, the number tied to the real
decision, jumps from 0.540 to 0.740. That gap is the argument for calling this lane a
ranking/scoring task: what matters lives in the ordering of the queue, not in the
classifier's raw separability.


## 2. Target or proxy

**Target: `final_refresh_score`, a continuous review-priority score used to rank pages — not a
single directly-observed column.** No one hand-labeled "review this page first," so the ranking
target is built, not observed. The starter pipeline stacks two proxies to get there:

1. `is_declining_label = (trend_direction == "down")` — the *internal* proxy the classifier
   trains on. It's explicitly a proxy, not a future observed outcome: it's a threshold rule
   computed from `trend_pct`, which is itself computed from the *same* 90-day window being
   scored — not "did this page's traffic keep falling over the next 30 days." The lane guide
   calls this out by name: *"a beginner proxy label... treat it that way, not as the ideal
   capstone target."*
2. `final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`
   — the actual lane output, blending that proxy-trained probability with the transparent rule.

Being honest about this changes what I can claim later: a page scoring high means *"the
observable signals in this window look like the signals of pages we're currently calling
declining,"* not *"this page will decline next month."* A stronger capstone target, per the lane
guide, would use a forward-looking window (`prior 90 days -> decline over next 30 days`) —
that's later work, not this notebook's job. `trend_direction` and `trend_pct` are therefore
never features, only the source of this proxy label — confirmed below.

In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

is_declining_label = (df["trend_direction"] == "down")
print("is_declining_label rate:", round(is_declining_label.mean(), 3))
print()
print(df["trend_direction"].value_counts())

print()
print("Forbidden as features (label source -- skills/flyrank/flyrank-data/SKILL.md):")
print("  - trend_direction   (defines is_declining_label directly)")
print("  - trend_pct         (trend_direction is computed from this)")

print()
print("final_refresh_score formula, for reference (docs/ml-intern-dataset-and-lane-guide.md):")
print("  final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)")


is_declining_label rate: 0.542

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Forbidden as features (label source -- skills/flyrank/flyrank-data/SKILL.md):
  - trend_direction   (defines is_declining_label directly)
  - trend_pct         (trend_direction is computed from this)

final_refresh_score formula, for reference (docs/ml-intern-dataset-and-lane-guide.md):
  final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)


## 3. Success metric

**Precision@50** — a top-K ranking metric: of the top 50 pages the queue ranks first, what
fraction are actually positive by the label? This is the metric the lane guide names directly
("Precision@50 if the team can act on 50 candidates") and the one my own `notebooks/01`/`02`
pipeline run was already validated against — on the initial split and on client-holdout data.

Why this metric and not accuracy or ROC AUC: those score the classifier as a classifier — they'd
reward a model that separates declining from stable pages well *anywhere* in the full 30,000-row
ranking. A reviewer only ever looks at the top of the list; a model that's excellent at telling
rank #4,000 from #4,001 apart but weak at the top 50 is useless here. Precision@K only asks about
the part of the ranking a human will actually see.

"Good" is directional, not a fixed passing number. My own committed pipeline shows Precision@50
= 0.240 (baseline) vs 0.740 (random forest) on the initial split, but only 0.620 vs 0.560 on
client-holdout (my own `w01` notebook-02 numbers) — the model's edge shrinks, and the simple rule
actually wins on unseen clients. So "good" here means: does the model beat the baseline by enough
margin, on clients it has never seen, to justify the added complexity — not an absolute number.

In [3]:
precision_at_50 = {
    "baseline_rules (hand rule)": {"initial_split": 0.240, "client_holdout": 0.620},
    "decision_tree":              {"initial_split": 0.540, "client_holdout": 0.560},
}

print(f"{'method':<30}{'initial split':>15}{'client holdout':>17}")
for name, vals in precision_at_50.items():
    print(f"{name:<30}{vals['initial_split']:>15.3f}{vals['client_holdout']:>17.3f}")

print()
print("On the initial split the tree already beats the hand rule (0.540 vs 0.240). On client")
print("holdout that edge nearly vanishes and the hand rule actually wins (0.620 vs 0.560, from")
print("my own w01 notebook). 'Good' can't just mean 'beats baseline on the split you tuned on' --")
print("it has to survive validation on clients the model has never seen.")


method                          initial split   client holdout
baseline_rules (hand rule)              0.240            0.620
decision_tree                           0.540            0.560

On the initial split the tree already beats the hand rule (0.540 vs 0.240). On client
holdout that edge nearly vanishes and the hand rule actually wins (0.620 vs 0.560, from
my own w01 notebook). 'Good' can't just mean 'beats baseline on the split you tuned on' --
it has to survive validation on clients the model has never seen.


## 4. The unit of analysis, as a real dataframe

One row = one published content page (`content_id`), aggregated over its trailing 90-day window,
for one client (`client_id`). 30,000 rows, 32 clients, no time series inside a row — the loaded
slice below shows this directly, plus a rough sketch of what the eventual target column looks
like (not the final scoring build — that's a later week's job).

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("shape:", df.shape)
print("unique content_id:", df["content_id"].nunique(), " (== row count -> one row per page)")
print("unique client_id:", df["client_id"].nunique())

cols_to_show = [
    "content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
    "ctr", "avg_position", "content_age_days", "days_since_last_update",
    "trend_direction", "trend_pct",
]
display(df[cols_to_show].head(5))

# --- Sketch of the target column (illustrative shape only, not the final scoring build) ---
# trend_direction/trend_pct appear here ONLY to construct this proxy label for illustration --
# they will not be used as model features later (see Section 2).
sketch = df[["content_id", "trend_direction", "trend_pct", "impressions_90d", "avg_position"]].copy()
sketch["is_declining_label_SKETCH"] = (sketch["trend_direction"] == "down").astype(int)

print()
print("Sketch of the target column's shape (a binary proxy that later feeds a continuous score):")
display(sketch.head(8))


shape: (30000, 44)
unique content_id: 30000  (== row count -> one row per page)
unique client_id: 32


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,days_since_last_update,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,keyword article,3803,29,0.76,10.6,187,20,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,7,0.05,20.3,445,25,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,11,0.09,36.5,141,20,down,-60.9
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,58,0.49,6.2,463,22,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,24,0.13,44.0,263,14,down,-34.7



Sketch of the target column's shape (a binary proxy that later feeds a continuous score):


,content_id,trend_direction,trend_pct,impressions_90d,avg_position,is_declining_label_SKETCH
0,content_304f48230142,down,-41.4,3803,10.6,1
1,content_a1fb4e703a9e,down,-57.7,15320,20.3,1
2,content_9aa793d4d895,down,-60.9,12581,36.5,1
3,content_331d6c4de07b,stable,-13.8,11751,6.2,0
4,content_d99b7a2d90ca,down,-34.7,19140,44.0,1
5,content_d4084a4bc775,down,-38.9,3970,8.5,1
6,content_9a34b442b552,down,-92.3,20,7.0,1
7,content_a63219c6e95a,stable,0.6,1724,21.2,0


## 5. Why ML beats a fixed rule here

Not because the four baseline signals are correlated with each other — I checked, and they
aren't (the cell below shows every pairwise correlation among them under 0.08). That actually
matters for the argument: a fixed-weight rule like the starter baseline
(`0.40*visibility + 0.30*freshness + 0.25*position + 0.05*depth_gap`) assumes each signal
contributes *independently and linearly*. Near-zero pairwise correlation doesn't mean the
signals don't interact — it means their interaction, if any, is conditional/nonlinear, which a
linear sum literally cannot represent and a correlation coefficient literally cannot detect.

The lane guide's own volume-floor warning is a concrete example of exactly that kind of
conditional effect: `position_tier == "top_3"` has a **median of only 3 impressions/90d** in
this dataset (checked below). A fixed rule scores every `top_3` page's `position_opportunity`
identically, whether it has 3 impressions or 30,000 — it has no way to say "position matters
*more* once there's enough volume behind it" unless someone manually writes that condition down.
A tree-based model finds exactly this kind of "signal A matters differently depending on signal
B" split on its own, without anyone having to anticipate and hand-code it.

And empirically, that's not theoretical: my own pipeline run shows a depth-limited decision tree
already beating the hand rule on the initial split (0.540 vs 0.240 Precision@50), and the random
forest going further (0.740) — real signal exists in interactions a short list of if-statements
can't express, even though (Section 3) that edge has to be re-earned on held-out clients before
I'd trust it.

In [5]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# The four signals behind the starter baseline's fixed-weight formula
signal_cols = ["avg_position", "ctr", "days_since_last_update", "impressions_90d"]
corr = df[signal_cols].corr(numeric_only=True).round(2)
print("Pairwise correlation among the baseline's four input signals:")
display(corr)
print()
print("Every off-diagonal value is <= 0.08 in magnitude -- essentially independent linearly.")
print("That's not evidence AGAINST an ML model helping -- a linear rule needs signals to combine")
print("additively to work well. Near-zero correlation just means any real interaction between")
print("these signals is conditional/nonlinear, which the fixed weighted sum can't represent at")
print("all, and a correlation coefficient can't see. The check below shows a concrete case.")

print()
# position_tier volume-floor check, from the lane guide's own warning
top3 = df[df["position_tier"] == "top_3"]
print(f"top_3 position_tier: {len(top3)} rows, median impressions_90d = {top3['impressions_90d'].median():.0f}")
print("A 3-impression page and a 30,000-impression page in this same tier get an IDENTICAL")
print("position_opportunity_score from the fixed rule -- it can't condition position's weight on")
print("volume unless someone manually hard-codes that as a fifth term. That's the kind of")
print("conditional split a tree-based model finds on its own.")


Pairwise correlation among the baseline's four input signals:


,avg_position,ctr,days_since_last_update,impressions_90d
avg_position,1.00,-0.07,0.07,-0.07
ctr,-0.07,1.00,-0.02,-0.02
days_since_last_update,0.07,-0.02,1.00,0.08
impressions_90d,-0.07,-0.02,0.08,1.00



Every off-diagonal value is <= 0.08 in magnitude -- essentially independent linearly.
That's not evidence AGAINST an ML model helping -- a linear rule needs signals to combine
additively to work well. Near-zero correlation just means any real interaction between
these signals is conditional/nonlinear, which the fixed weighted sum can't represent at
all, and a correlation coefficient can't see. The check below shows a concrete case.

top_3 position_tier: 2321 rows, median impressions_90d = 3
A 3-impression page and a 30,000-impression page in this same tier get an IDENTICAL
position_opportunity_score from the fixed rule -- it can't condition position's weight on
volume unless someone manually hard-codes that as a fifth term. That's the kind of
conditional split a tree-based model finds on its own.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.